# Experiments

- Start `mlflow` on terminal

```bash
mlflow ui \
  --backend-store-uri sqlite:///../mlflow_db/mlflow.db \
  --default-artifact-root ../mlartifacts \
  --host 127.0.0.1 \
  --port 5001
```

In [1]:
# basic setup and import
import os
import json
import tempfile
import numpy as np
import pandas as pd

import mlflow
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score

# ----------------------------
# MLflow experiment
# ----------------------------
mlflow.set_tracking_uri("sqlite:///../mlflow_db/mlflow.db")
print("Tracking URI:", mlflow.get_tracking_uri())


/usr/local/anaconda3/envs/ml_kaggle/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/anaconda3/envs/ml_kaggle/lib/python3.11/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Tracking URI: sqlite:///../mlflow_db/mlflow.db


In [2]:
mlflow.set_experiment("heart_disease_kaggle")

2026/02/09 12:54:42 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/02/09 12:54:42 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/02/09 12:54:42 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/02/09 12:54:42 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/02/09 12:54:42 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/02/09 12:54:42 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/02/09 12:54:43 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/09 12:54:43 INFO alembic.runtime.migration: Will assume non-transactional DDL.


<Experiment: artifact_location='/Users/gabi/codes/kaggle/02_predicting_heart_disease/mlruns/1', creation_time=1770636741299, experiment_id='1', last_update_time=1770636741299, lifecycle_stage='active', name='heart_disease_kaggle', tags={}>

In [3]:
# treat dataset
def treat_dataset(data):
    # rename column names for easier access
    data.columns = [col.lower().replace(' ', '_') for col in data.columns]

    # convert heart_disease to binary
    data['heart_disease'] = data['heart_disease'].apply(lambda x: 0 if x == "Absence" else 1)

    return data


In [4]:
# read data file and prepare dataset
target = "heart_disease"

data = pd.read_csv("data/train.csv")

train_data = treat_dataset(data)

## Set features

In [5]:
# set features
numeric_features = ["age", "bp", "cholesterol", "max_hr", "st_depression", "number_of_vessels_fluro"]
categorical_features = ["sex", "fbs_over_120", "exercise_angina", "chest_pain_type", "ekg_results", "slope_of_st", "thallium"]

features = numeric_features + categorical_features


## CatBoost Experimentation

In [ ]:
from catboost import CatBoostClassifier, Pool

# ----------------------------
# Data prep: drop id, split holdout once
# ----------------------------
df = train_data.drop(columns=["id"]).copy()
X = df[features].copy()
y = df[target].astype(int).copy()

cat_idx = [X.columns.get_loc(c) for c in categorical_features]

X_tr, X_ho, y_tr, y_ho = train_test_split(
    X, y,
    test_size=0.15,
    stratify=y,
    random_state=42
)


# ----------------------------
# CV evaluation function (returns fold metrics + best iters)
# ----------------------------
def cv_catboost_auc(params, X_train, y_train, cat_idx, n_splits=5, seed=42):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    fold_aucs = []
    fold_best_iters = []

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X_train, y_train), 1):
        X_trf, X_vaf = X_train.iloc[tr_idx], X_train.iloc[va_idx]
        y_trf, y_vaf = y_train.iloc[tr_idx], y_train.iloc[va_idx]

        train_pool = Pool(X_trf, y_trf, cat_features=cat_idx)
        valid_pool = Pool(X_vaf, y_vaf, cat_features=cat_idx)

        model = CatBoostClassifier(
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=seed,
            allow_writing_files=False,
            verbose=False,
            od_type="Iter",
            od_wait=int(params["od_wait"]),
            iterations=int(params["iterations"]),
            learning_rate=float(params["learning_rate"]),
            depth=int(params["depth"]),
            l2_leaf_reg=float(params["l2_leaf_reg"]),
            subsample=float(params["subsample"]),
            rsm=float(params["rsm"]),
            min_data_in_leaf=int(params["min_data_in_leaf"]), # Optional: if you want more regularization control
        )

        model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

        p = model.predict_proba(valid_pool)[:, 1]
        auc = roc_auc_score(y_vaf, p)

        fold_aucs.append(auc)
        fold_best_iters.append(model.get_best_iteration())

    return fold_aucs, fold_best_iters

# ----------------------------
# Hyperopt search space
# ----------------------------
space = {
    # big upper bound; early stopping selects best iteration per fold
    "iterations": hp.quniform("iterations", 1000, 6000, 250),
    "learning_rate": hp.loguniform("learning_rate", np.log(0.01), np.log(0.2)),
    "depth": hp.quniform("depth", 4, 10, 1),
    "l2_leaf_reg": hp.loguniform("l2_leaf_reg", np.log(1.0), np.log(20.0)),
    "subsample": hp.uniform("subsample", 0.6, 1.0),
    "rsm": hp.uniform("rsm", 0.6, 1.0),
    "od_wait": hp.quniform("od_wait", 100, 400, 50),
    "min_data_in_leaf": hp.quniform("min_data_in_leaf", 5, 80, 5),
}

# ----------------------------
# Objective function: logs each trial into MLflow
# ----------------------------
def objective(params):
    # Cast hyperopt outputs
    params = dict(params)
    params["iterations"] = int(params["iterations"])
    params["depth"] = int(params["depth"])
    params["od_wait"] = int(params["od_wait"])
    params["min_data_in_leaf"] = int(params["min_data_in_leaf"])

    with mlflow.start_run(nested=True):
        # Log params
        mlflow.log_params(params)

        # Log feature config as artifact (json)
        feat_payload = {
            "numeric_features": numeric_features,
            "categorical_features": categorical_features,
            "all_features": features
        }
        with tempfile.TemporaryDirectory() as tmpdir:
            feat_path = os.path.join(tmpdir, "features.json")
            with open(feat_path, "w") as f:
                json.dump(feat_payload, f, indent=2)
            mlflow.log_artifact(feat_path, artifact_path="config")

        # CV
        fold_aucs, fold_best_iters = cv_catboost_auc(params, X_tr, y_tr, cat_idx, n_splits=10, seed=42)
        mean_auc = float(np.mean(fold_aucs))
        std_auc = float(np.std(fold_aucs))
        med_best_iter = int(np.median(fold_best_iters))

        # Log CV metrics
        mlflow.log_metric("cv_auc_mean", mean_auc)
        mlflow.log_metric("cv_auc_std", std_auc)
        mlflow.log_metric("cv_best_iter_median", med_best_iter)

        for i, auc in enumerate(fold_aucs, 1):
            mlflow.log_metric(f"cv_auc_fold_{i}", float(auc))
        for i, bi in enumerate(fold_best_iters, 1):
            mlflow.log_metric(f"cv_best_iter_fold_{i}", int(bi))

        # Train final model on full TRAIN split using median best_iter
        train_pool_full = Pool(X_tr, y_tr, cat_features=cat_idx)
        holdout_pool = Pool(X_ho, y_ho, cat_features=cat_idx)

        final_model = CatBoostClassifier(
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=42,
            allow_writing_files=False,
            verbose=False,
            iterations=med_best_iter,
            learning_rate=float(params["learning_rate"]),
            depth=int(params["depth"]),
            l2_leaf_reg=float(params["l2_leaf_reg"]),
            subsample=float(params["subsample"]),
            rsm=float(params["rsm"]),
        )

        final_model.fit(train_pool_full)

        # Holdout AUC
        p_ho = final_model.predict_proba(holdout_pool)[:, 1]
        ho_auc = float(roc_auc_score(y_ho, p_ho))

        mlflow.log_metric("holdout_auc", ho_auc)

        # Save model artifact
        with tempfile.TemporaryDirectory() as tmpdir:
            model_path = os.path.join(tmpdir, "catboost_model.cbm")
            final_model.save_model(model_path)
            mlflow.log_artifact(model_path, artifact_path="model")

        # Hyperopt minimizes, so return negative AUC
        return {"loss": -mean_auc, "status": STATUS_OK, "cv_auc_mean": mean_auc, "holdout_auc": ho_auc}

# ----------------------------
# Run the search (top-level MLflow run)
# ----------------------------
max_evals = 25
trials = Trials()

with mlflow.start_run(run_name="catboost_hyperopt"):
    mlflow.log_param("max_evals", max_evals)
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=max_evals,
        trials=trials,
        rstate=np.random.default_rng(42),
    )

print("Best hyperopt space (raw):", best)

# Optional: extract the best trial details
best_trial = min(trials.results, key=lambda r: r["loss"])
print("Best CV AUC:", best_trial["cv_auc_mean"])
print("Best Holdout AUC:", best_trial["holdout_auc"])


  4%|▍         | 1/25 [1:47:27<42:58:49, 6447.05s/trial, best loss: -0.9552852397565548]